Step 1: Install Required Libraries

In [11]:
# Installing required dependencies
!pip install --upgrade transformers huggingface_hub accelerat

Step 2: Import Required Libraries

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

Step 3: Load Pre-trained Transformer Model and Tokenizer

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


print("Model loaded successfully!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded successfully!


Step 4: Define Chatbot Response Function

In [8]:
# Creating a function for generating chatbot responses
def generate_response(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Converting text into model input tensors
    model_inputs = tokenizer(
        [text],
        return_tensors="pt"
    ).to(model.device)

    # Generating response tokens from the model
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

    # Extracting only newly generated tokens (excluding input tokens)
    generated_ids = [output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    # Decoding tokens into readable text
    response = tokenizer.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return response


Step 5: Build Interactive Chatbot Loop

In [10]:
# Starting chatbot interaction loop
print("Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.")

# Initializing conversation history with system instruction
messages = [
    {"role": "system", "content": "You are a helpful and friendly AI assistant."}
]

while True:
    # Taking user input
    user_input = input("You: ")

    # Checking for exit condition
    if user_input.lower() in ["exit", "quit"]:
        print("Chatbot: Goodbye! Have a great day.")
        break

    # Adding user message to conversation history
    messages.append({
        "role": "user",
        "content": user_input
    })

    # Generating chatbot response based on conversation
    response = generate_response(messages)

    print("Chatbot:", response)
    # Storing chatbot reply for maintaining context
    messages.append({
        "role": "assistant",
        "content": response
    })

Chatbot: Hello! I am your AI assistant. Type 'exit' or 'quit' to stop.


You:  Hi


Chatbot: Hello! How can I help you today? Is there something specific you would like to talk about or ask me? I'm here to assist with any questions or concerns you may have.


You:  Can you please tell me what is Data science


Chatbot: Data science is a field of study that focuses on the use of statistical, mathematical, and computational methods to extract knowledge and insights from large datasets. It involves the analysis of data to identify patterns, trends, and relationships, which can be used for decision-making and problem-solving.

Data scientists often work with different types of data such as text, images, audio, and video, and they use various programming


You:  Thank you for your response


Chatbot: You're welcome! If you have any more questions, feel free to ask. I'm here to help in any way I can.


You:  exit


Chatbot: Goodbye! Have a great day.
